# Suppressed-activation intervention: paired Base / Causal-intervention examples

Two evidence layers, both reading saved artifacts (CPU, no model run):
1. DEVELOPMENT regression replays (heavily exposed prompts): the frozen L20 C1.5 candidate
   transfers naming and leg-count with persistent identity on both animals; property fails.
2. FRESH evaluation (12 never-executed questions, frozen manifest rev4): candidate 6/12
   primary semantic success (answer + persistent identity + coherence) vs 1/12 for its
   strength-matched random control. A descriptive rate on a paired convenience sample -
   limited measured reliability after real attempts, NOT a solved mechanism.

In [1]:
import json
from pathlib import Path

BATCH = Path("/workspace/2026/suppressed-activations-batchwork")  # worktree holding the measured out/

# Slot -> result.json path (relative to BATCH). Edit to swap development evidence for new runs.
# Replay artifacts (regression verification, tasks 1005/1006): the same measured cells
# re-run from the recovered config; probabilities are bitwise-identical to the originals.
PATHS = {
    "legs_dog_c15": "out/2026-09-10_replay-C1.5-legs-dog/result.json",
    "legs_ant_c15": "out/2026-09-10_replay-C1.5-legs-ant/result.json",
    "naming_dog_c15": "out/2026-09-10_replay-C1.5-naming-dog/result.json",
    "naming_ant_c15": "out/2026-09-10_replay-C1.5-naming-ant/result.json",
    "prop_dog_c15": "out/2026-09-10_replay-C1.5-property-dog/result.json",
    "prop_ant_c15": "out/2026-09-10_replay-C1.5-property-ant/result.json",
}
DB = {}
for slot, rel in PATHS.items():
    if rel is None:
        continue
    p = BATCH / rel
    assert p.exists(), f"missing {rel}"
    data = json.loads(p.read_text())
    row = data["rows"][0]
    DB[slot] = (data, row)
print("loaded measured slots:", sorted(DB.keys()))

loaded measured slots: ['legs_ant_c15', 'legs_dog_c15', 'naming_ant_c15', 'naming_dog_c15', 'prop_ant_c15', 'prop_dog_c15']


## Tokenizer to decode exact previews

In [2]:
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer

data0 = DB["legs_dog_c15"][0]
TOK = AutoTokenizer.from_pretrained("Qwen/" + data0["model"].split("/")[-1],
                                     revision=data0["revision"], trust_remote_code=True)
print("tokenizer:", data0["model"], data0["revision"][:12])

def md_table(rows, with_delta):
    lines = ["| token | log p | p |" + (" change in log p |" if with_delta else ""),
             "|---|---|---|" + ("---|" if with_delta else "")]
    for r in rows[:10]:
        cell = [f"`{r['token']}`", f"{r['logp']:.3f}", f"{r['p']:.5f}"] + ([f"{r['delta_logp']:+.3f}"] if with_delta else [])
        lines.append("| " + " | ".join(cell) + " |")
    return "\n".join(lines)

def first32(row, kind):
    ids = row[kind]["token_ids"]
    return len(ids), TOK.decode(ids[:32])

from IPython.display import Markdown, display

def block(slot, title):
    data, row = DB[slot]
    art = BATCH / PATHS[slot].replace("result.json", row["log"])
    nbase, pbase = first32(row, "base_generation")
    nsteer, psteer = first32(row, "generation")
    rendered_s = data["rendered_inputs"]["source"]["rendered"]
    rendered_d = data["rendered_inputs"]["donor"]["rendered"]
    display(Markdown(f"## {title} :: `{row['condition_id']}`"))
    display(Markdown(
        f"Source rendered model input (`repr`, chat wrapper included):\n\n"
        f"```text\n{rendered_s!r}\n```\n\n"
        f"Donor rendered model input (`repr`):\n\n```text\n{rendered_d!r}\n```"))
    display(Markdown(
        f"### Base\n\n"
        f"Clean-source prefill readout (**unvalidated**, verbatim): "
        f"`{json.dumps(row['base_readout'], ensure_ascii=False)}`\n\n"
        f"First 32 of {nbase} generated tokens, decoded:\n\n```text\n{pbase}\n```\n\n"
        f"Full Base continuation ({nbase} tokens): [{art.name}]({art})\n\n"
        f"Base top-10:\n\n{md_table(row['base_top_tokens'], with_delta=False)}"))
    display(Markdown(
        f"### Causal intervention\n\n"
        f"Edited-source prefill readout (**unvalidated**, verbatim): "
        f"`{json.dumps(row['readout'], ensure_ascii=False)}`\n\n"
        f"Final-decode readout (**unvalidated**; the state the last token was written from): "
        f"`{json.dumps(row['last_decode_readout'], ensure_ascii=False)}`\n\n"
        f"First 32 of {nsteer} generated tokens, decoded:\n\n```text\n{psteer}\n```\n\n"
        f"Full intervention continuation ({nsteer} tokens): [{art.name}]({art})\n\n"
        f"Intervention top-10 (change in log p vs Base):\n\n"
        f"{md_table(row['top_tokens'], with_delta=True)}\n\n"
        f"Expected base `{row['expected_base_answer']}` steered `{row['expected_steered_answer']}`; "
        f"S_swap={row['swap_log_odds_shift']:+.3f}, bare_answer_mass={row['bare_answer_mass']:.4f}, "
        f"repeat_bigrams={row['repeated_bigram_fraction']:.3f}"))

tokenizer: Qwen/Qwen3.5-4B 851bf6e806ef


## Clean source/donor calibration (competence reporting; readouts unvalidated)

In [3]:
cal = ["| slot | clean source answer | donor first answer | donor p(target) | donor readout (unvalidated) |",
       "|---|---|---|---|---|"]
for slot in PATHS:
    data, row = DB[slot]
    dgen = row.get("donor_generation", {}).get("text", "")
    src_answer = row.get("expected_base_answer")
    cal.append(f"| {slot} | {src_answer} | {row.get('donor_first_answer')} "
               f"| {row.get('donor_p_target'):.4f} "
               f"| `{json.dumps(row.get('target_readout', []), ensure_ascii=False)}` |")
display(Markdown(
    "Clean source/donor competence from the full saved generations (first answer where the "
    "parser found one; donor readout is the prefill suppression readout, unvalidated):\n\n"
    + "\n".join(cal)))
# donor generations are the semantic competence evidence; show one excerpt per slot
excerpts = ["| slot | clean donor continuation (first 140 chars) |", "|---|---|"]
for slot in PATHS:
    data, row = DB[slot]
    dgen = row.get("donor_generation", {}).get("text", "").replace("\n", " ")[:140]
    excerpts.append(f"| {slot} | {dgen} |")
display(Markdown("\n".join(excerpts)))

Clean source/donor competence from the full saved generations (first answer where the parser found one; donor readout is the prefill suppression readout, unvalidated):

| slot | clean source answer | donor first answer | donor p(target) | donor readout (unvalidated) |
|---|---|---|---|---|
| legs_dog_c15 | 8 | 4 | 0.9886 | `[" respuesta", "拥有着", "getResponse", " risposta", "_response", " response", " respond", "responses"]` |
| legs_ant_c15 | 8 | 6 | 0.9313 | `[" respuesta", " risposta", "_response", " response", "getResponse", " respond", " resposta", "-road"]` |
| naming_dog_c15 | Spider | None | 0.0063 | `[" respuesta", "谜底", "anjangan", "responseObject", "getResponse", " response", " respond", "大名"]` |
| naming_ant_c15 | Spider | None | 0.0022 | `[" respuesta", "getResponse", "ưởi", "responseObject", "谜底", "_response", " respuestas", " response"]` |
| prop_dog_c15 | No | None | 0.0018 | `[" risposta", " respuesta", " respuestas", "getResponse", " cpt", " resposta", " response", " respond"]` |
| prop_ant_c15 | No | None | 0.0007 | `[" risposta", " respuesta", "λευτα", " resposta", " réponses", " respuestas", " پاسخ", "NON"]` |

| slot | clean donor continuation (first 140 chars) |
|---|---|
| legs_dog_c15 | 4  The animal you are referring to is a dog, which is a domesticated carnivorous mammal known for its loyalty and versatility. Dogs typicall |
| legs_ant_c15 | 6  The animal described is an ant, which is a social insect known for living in large, organized colonies. These insects communicate primari |
| naming_dog_c15 | 狗 (Dog)  The dog is a domesticated carnivorous mammal that has been bred by humans for thousands of years. They are known for their loyalty, |
| naming_ant_c15 | 蚂蚁 (Ant)  The ant is a small, social insect known for living in large, organized colonies. They communicate with each other primarily throug |
| prop_dog_c15 | 1. Yes, the dog is a mammal. 2. Dogs are warm-blooded vertebrates that possess hair or fur, which helps them regulate their body temperature |
| prop_ant_c15 | 1. No, it is not a mammal. 2. It is an insect, specifically an ant. 3. Ants are known for their complex social structures and ability to com |

## Legs (C=1.5): digit + identity transfer (measured)

In [4]:
block("legs_dog_c15", "Dog legs, C=1.5")
block("legs_ant_c15", "Ant legs, C=1.5")

## Dog legs, C=1.5 :: `003_span_correction_sweep_C1.5`

Source rendered model input (`repr`, chat wrapper included):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: How many legs does the animal that spins webs have?\nAnswer: '
```

Donor rendered model input (`repr`):

```text
"<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: How many legs does the animal that barks and is called man's best friend have?\nAnswer: "
```

### Base

Clean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", " risposta", "getResponse", "拥有着", " resposta", "response", "_response", " respond"]`

First 32 of 58 generated tokens, decoded:

```text
8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect
```

Full Base continuation (58 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-legs-dog/conditions/003_span_correction_sweep_C1.5/run.md)

Base top-10:

| token | log p | p |
|---|---|---|
| `8` | -0.084 | 0.91916 |
| `4` | -3.084 | 0.04576 |
| `6` | -3.584 | 0.02776 |
| `3` | -6.084 | 0.00228 |
| `1` | -6.334 | 0.00177 |
| `2` | -6.709 | 0.00122 |
| `0` | -7.459 | 0.00058 |
| `5` | -7.834 | 0.00040 |
| `7` | -7.959 | 0.00035 |
| `Eight` | -8.584 | 0.00019 |

### Causal intervention

Edited-source prefill readout (**unvalidated**, verbatim): `[" contradictions", " respuesta", " risposta", " pia", "向记者", " rendimiento", "getResponse", "renders"]`

Final-decode readout (**unvalidated**; the state the last token was written from): `["卡和", "这个大", " spal", " عاط", "Mirror", "ЗИ", "ネットで", "错过的"]`

First 32 of 65 generated tokens, decoded:

```text
4

The animal is a dog, which is a domesticated canine known for its loyalty and ability to understand human commands. Dogs typically have a short history of
```

Full intervention continuation (65 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-legs-dog/conditions/003_span_correction_sweep_C1.5/run.md)

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `4` | -0.032 | 0.96866 | +3.052 |
| `2` | -3.657 | 0.02581 | +3.052 |
| ` Four` | -6.657 | 0.00129 | +3.865 |
| `1` | -7.032 | 0.00088 | -0.698 |
| `8` | -7.282 | 0.00069 | -7.198 |
| `0` | -7.407 | 0.00061 | +0.052 |
| ` The` | -7.782 | 0.00042 | +3.865 |
| ` Dogs` | -7.782 | 0.00042 | +9.646 |
| `3` | -7.907 | 0.00037 | -1.823 |
| `6` | -8.594 | 0.00019 | -5.010 |

Expected base `8` steered `4`; S_swap=+10.250, bare_answer_mass=0.9693, repeat_bigrams=0.016

## Ant legs, C=1.5 :: `003_span_correction_sweep_C1.5`

Source rendered model input (`repr`, chat wrapper included):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: How many legs does the animal that spins webs have?\nAnswer: '
```

Donor rendered model input (`repr`):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: How many legs does the animal that lives in colonies and follows pheromone trails have?\nAnswer: '
```

### Base

Clean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", " risposta", "getResponse", "拥有着", " resposta", "response", "_response", " respond"]`

First 32 of 58 generated tokens, decoded:

```text
8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect
```

Full Base continuation (58 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-legs-ant/conditions/003_span_correction_sweep_C1.5/run.md)

Base top-10:

| token | log p | p |
|---|---|---|
| `8` | -0.084 | 0.91916 |
| `4` | -3.084 | 0.04576 |
| `6` | -3.584 | 0.02776 |
| `3` | -6.084 | 0.00228 |
| `1` | -6.334 | 0.00177 |
| `2` | -6.709 | 0.00122 |
| `0` | -7.459 | 0.00058 |
| `5` | -7.834 | 0.00040 |
| `7` | -7.959 | 0.00035 |
| `Eight` | -8.584 | 0.00019 |

### Causal intervention

Edited-source prefill readout (**unvalidated**, verbatim): `[" respuesta", " risposta", "梦想的", "\":@\"", "getResponse", " piccolo", " persistence", "debit"]`

Final-decode readout (**unvalidated**; the state the last token was written from): `["微小的", " pequeños", " کوچک", " cari", "small", " pequeña", " мелкие", " pequeño"]`

First 32 of 65 generated tokens, decoded:

```text
6

The ant is a small, hardworking insect known for its ability to carry objects much larger than itself. These social creatures live in vast colonies and communicate
```

Full intervention continuation (65 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-legs-ant/conditions/003_span_correction_sweep_C1.5/run.md)

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `6` | -0.018 | 0.98181 | +3.566 |
| `8` | -4.643 | 0.00963 | -4.559 |
| `3` | -5.268 | 0.00515 | +0.816 |
| `1` | -6.518 | 0.00148 | -0.184 |
| `4` | -7.268 | 0.00070 | -4.184 |
| `2` | -7.893 | 0.00037 | -1.184 |
| `7` | -8.643 | 0.00018 | -0.684 |
| ` Six` | -8.768 | 0.00016 | +2.816 |
| `0` | -9.268 | 0.00009 | -1.809 |
| `5` | -9.393 | 0.00008 | -1.559 |

Expected base `8` steered `6`; S_swap=+8.125, bare_answer_mass=0.9914, repeat_bigrams=0.000

## Naming (C=1.5): identity transfer (measured)

In [5]:
block("naming_dog_c15", "Dog naming, C=1.5")
block("naming_ant_c15", "Ant naming, C=1.5")

## Dog naming, C=1.5 :: `003_span_correction_sweep_C1.5`

Source rendered model input (`repr`, chat wrapper included):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: What is the animal that spins webs called?\nAnswer: '
```

Donor rendered model input (`repr`):

```text
"<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: What is the animal that barks and is called man's best friend called?\nAnswer: "
```

### Base

Clean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", "responseObject", " respond", " respuestas", "getResponse", "谜底", "_response", " cries"]`

First 32 of 74 generated tokens, decoded:

```text
蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in
```

Full Base continuation (74 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-naming-dog/conditions/003_span_correction_sweep_C1.5/run.md)

Base top-10:

| token | log p | p |
|---|---|---|
| `蜘蛛` | -0.404 | 0.66750 |
| `蛛` | -2.279 | 0.10236 |
| ` Spider` | -2.529 | 0.07972 |
| `Spider` | -2.967 | 0.05147 |
| ` spider` | -3.779 | 0.02284 |
| ` The` | -4.592 | 0.01014 |
| `8` | -5.154 | 0.00578 |
| ` **` | -5.279 | 0.00510 |
| ` spiders` | -5.279 | 0.00510 |
| `1` | -5.529 | 0.00397 |

### Causal intervention

Edited-source prefill readout (**unvalidated**, verbatim): `[" respuesta", "ErrorResponse", "@Xml", " faj", " ARC", "responseObject", " sụt", ".Mar"]`

Final-decode readout (**unvalidated**; the state the last token was written from): `["卡和", "Mirror", " عاط", " предна", "Dummy", "这个大", "グレ", "ツイート"]`

First 32 of 57 generated tokens, decoded:

```text
狗 (Dog)

The dog is a domesticated canine that has lived alongside humans for thousands of years. They are known for their loyalty, intelligence, and
```

Full intervention continuation (57 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-naming-dog/conditions/003_span_correction_sweep_C1.5/run.md)

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `狗` | -0.457 | 0.63326 | +10.947 |
| `犬` | -2.332 | 0.09711 | +10.791 |
| ` Dog` | -3.082 | 0.04587 | +11.979 |
| `狗狗` | -3.082 | 0.04587 | +10.822 |
| ` The` | -3.519 | 0.02962 | +1.072 |
| ` A` | -3.894 | 0.02036 | +2.510 |
| `猫` | -4.269 | 0.01399 | +5.729 |
| `1` | -4.644 | 0.00962 | +0.885 |
| `一只` | -4.644 | 0.00962 | +1.697 |
| `Dog` | -5.082 | 0.00621 | +10.354 |

Expected base `Spider` steered `Dog`; S_swap=+24.357, bare_answer_mass=0.0062, repeat_bigrams=0.000

## Ant naming, C=1.5 :: `003_span_correction_sweep_C1.5`

Source rendered model input (`repr`, chat wrapper included):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: What is the animal that spins webs called?\nAnswer: '
```

Donor rendered model input (`repr`):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: What is the animal that lives in colonies and follows pheromone trails called?\nAnswer: '
```

### Base

Clean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", "responseObject", " respond", " respuestas", "getResponse", "谜底", "_response", " cries"]`

First 32 of 74 generated tokens, decoded:

```text
蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in
```

Full Base continuation (74 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-naming-ant/conditions/003_span_correction_sweep_C1.5/run.md)

Base top-10:

| token | log p | p |
|---|---|---|
| `蜘蛛` | -0.404 | 0.66750 |
| `蛛` | -2.279 | 0.10236 |
| ` Spider` | -2.529 | 0.07972 |
| `Spider` | -2.967 | 0.05147 |
| ` spider` | -3.779 | 0.02284 |
| ` The` | -4.592 | 0.01014 |
| `8` | -5.154 | 0.00578 |
| ` **` | -5.279 | 0.00510 |
| ` spiders` | -5.279 | 0.00510 |
| `1` | -5.529 | 0.00397 |

### Causal intervention

Edited-source prefill readout (**unvalidated**, verbatim): `["-push", "に近い", " respuesta", "是何含义", "leaders", "remento", "粮食", "顶着"]`

Final-decode readout (**unvalidated**; the state the last token was written from): `["微小的", " mikro", " Малень", " KAR", "卡和", "small", " cari", "แคร"]`

First 32 of 63 generated tokens, decoded:

```text
 Ant

Description:
The ant is a small, social insect known for its ability to live in large colonies.
They are famous for their incredible strength,
```

Full intervention continuation (63 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-naming-ant/conditions/003_span_correction_sweep_C1.5/run.md)

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| ` Ant` | -1.734 | 0.17654 | +9.326 |
| `蚂蚁` | -1.859 | 0.15580 | +6.858 |
| ` The` | -2.234 | 0.10708 | +2.358 |
| `1` | -2.359 | 0.09449 | +3.170 |
| ` **` | -2.922 | 0.05384 | +2.358 |
| ` Worker` | -3.047 | 0.04751 | +11.576 |
| `蜜蜂` | -3.234 | 0.03939 | +5.045 |
| `答案` | -3.484 | 0.03068 | +4.358 |
| `工` | -3.984 | 0.01861 | +6.139 |
| `蚁` | -4.109 | 0.01642 | +4.920 |

Expected base `Spider` steered `Ant`; S_swap=+13.812, bare_answer_mass=0.0071, repeat_bigrams=0.033

## Property (C=1.5): measured FAILURE, kept visible

The answer stays `No` for both animals while identity moves. Dog: "The animal that spins
webs is a dog, which is a mammal" -- then answers No, a self-contradiction (a dog is a
mammal). Ant: identity moves to a honey bee (a third animal). Property transfer does not
work at the frozen operating point; these rows are shown unedited.

In [6]:
block("prop_dog_c15", "Dog property, C=1.5 (FAILED transfer: identity moves, answer contradicts)")
block("prop_ant_c15", "Ant property, C=1.5 (FAILED transfer: identity moves to honey bee)")

## Dog property, C=1.5 (FAILED transfer: identity moves, answer contradicts) :: `003_span_correction_sweep_C1.5`

Source rendered model input (`repr`, chat wrapper included):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: Is the animal that spins webs a mammal?\nAnswer: '
```

Donor rendered model input (`repr`):

```text
"<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: Is the animal that barks and is called man's best friend a mammal?\nAnswer: "
```

### Base

Clean-source prefill readout (**unvalidated**, verbatim): `[" risposta", " respuesta", " tidak", "λευτα", " respuestas", " resposta", " нек", " réponses"]`

First 32 of 85 generated tokens, decoded:

```text
 No.

The animal that spins webs is a spider, which belongs to the class Arachnida rather than the class Mammalia. Unlike mammals,
```

Full Base continuation (85 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-property-dog/conditions/003_span_correction_sweep_C1.5/run.md)

Base top-10:

| token | log p | p |
|---|---|---|
| ` No` | -1.211 | 0.29792 |
| `1` | -1.586 | 0.20476 |
| `<think>` | -1.961 | 0.14073 |
| ` Yes` | -2.711 | 0.06648 |
| ` **` | -3.023 | 0.04863 |
| `0` | -3.086 | 0.04569 |
| `2` | -3.648 | 0.02603 |
| `否` | -3.711 | 0.02445 |
| `3` | -3.836 | 0.02158 |
| `5` | -3.961 | 0.01905 |

### Causal intervention

Edited-source prefill readout (**unvalidated**, verbatim): `[" risposta", " respuesta", " respuestas", " resposta", "','=',$", "มาเป็น", " pyt", " réponses"]`

Final-decode readout (**unvalidated**; the state the last token was written from): `["卡和", "ЗИ", " عاط", "Mirror", "镜头里", " Servi", "这个大", "ネットで"]`

First 32 of 48 generated tokens, decoded:

```text
 No.

The animal that spins webs is a dog, which is a mammal. Dogs are known for their loyalty and ability to understand human commands. They
```

Full intervention continuation (48 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-property-dog/conditions/003_span_correction_sweep_C1.5/run.md)

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| ` No` | -0.988 | 0.37231 | +0.223 |
| `1` | -1.863 | 0.15520 | -0.277 |
| ` Yes` | -1.988 | 0.13697 | +0.723 |
| `<think>` | -2.613 | 0.07331 | -0.652 |
| `0` | -3.238 | 0.03924 | -0.152 |
| `2` | -3.301 | 0.03686 | +0.348 |
| `否` | -3.426 | 0.03253 | +0.285 |
| ` **` | -3.488 | 0.03056 | -0.465 |
| `5` | -3.738 | 0.02380 | +0.223 |
| `4` | -4.176 | 0.01537 | +0.348 |

Expected base `No` steered `Yes`; S_swap=+0.063, bare_answer_mass=0.0013, repeat_bigrams=0.022

## Ant property, C=1.5 (FAILED transfer: identity moves to honey bee) :: `003_span_correction_sweep_C1.5`

Source rendered model input (`repr`, chat wrapper included):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: Is the animal that spins webs a mammal?\nAnswer: '
```

Donor rendered model input (`repr`):

```text
'<|im_start|>user\nAnswer the question with the answer first. Then describe the animal in three sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nQuestion: Is the animal that lives in colonies and follows pheromone trails a mammal?\nAnswer: '
```

### Base

Clean-source prefill readout (**unvalidated**, verbatim): `[" risposta", " respuesta", " tidak", "λευτα", " respuestas", " resposta", " нек", " réponses"]`

First 32 of 85 generated tokens, decoded:

```text
 No.

The animal that spins webs is a spider, which belongs to the class Arachnida rather than the class Mammalia. Unlike mammals,
```

Full Base continuation (85 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-property-ant/conditions/003_span_correction_sweep_C1.5/run.md)

Base top-10:

| token | log p | p |
|---|---|---|
| ` No` | -1.211 | 0.29792 |
| `1` | -1.586 | 0.20476 |
| `<think>` | -1.961 | 0.14073 |
| ` Yes` | -2.711 | 0.06648 |
| ` **` | -3.023 | 0.04863 |
| `0` | -3.086 | 0.04569 |
| `2` | -3.648 | 0.02603 |
| `否` | -3.711 | 0.02445 |
| `3` | -3.836 | 0.02158 |
| `5` | -3.961 | 0.01905 |

### Causal intervention

Edited-source prefill readout (**unvalidated**, verbatim): `[" risposta", "(push", " respuesta", " Kraft", "[__", "(trace", "small", "を目"]`

Final-decode readout (**unvalidated**; the state the last token was written from): `["すり", " Малень", " piccoli", " piccolo", " pequeños", "のエ", "微小的", "微电子"]`

First 32 of 128 generated tokens, decoded:

```text
 No.

The animal that spins webs is a honey bee, which is a social insect known for its role in agriculture and pollination. Unlike mammals, bees
```

Full intervention continuation (128 tokens): [run.md](/workspace/2026/suppressed-activations-batchwork/out/2026-09-10_replay-C1.5-property-ant/conditions/003_span_correction_sweep_C1.5/run.md)

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| ` No` | -0.439 | 0.64477 | +0.772 |
| ` **` | -2.314 | 0.09888 | +0.710 |
| `1` | -2.439 | 0.08726 | -0.853 |
| `<think>` | -3.064 | 0.04671 | -1.103 |
| ` Yes` | -3.189 | 0.04122 | -0.478 |
| `0` | -4.064 | 0.01718 | -0.978 |
| `3` | -4.501 | 0.01109 | -0.665 |
| `2` | -4.751 | 0.00864 | -1.103 |
| `5` | -5.126 | 0.00594 | -1.165 |
| `4` | -5.501 | 0.00408 | -0.978 |

Expected base `No` steered `Yes`; S_swap=-1.062, bare_answer_mass=0.0002, repeat_bigrams=0.032

## Provenance

Each artifact stores its own git describe, code SHA-256 of the runner files, resolved config,
and revision; the runner writes them at run time and the reliability table must be computed
from those fields, not from notebook state. Worker: PI[claude] (glm-5p3-flash session,
user-confirmed switch). Recovery decision: batchwork `slop/2026-09-10_recovery_decision.md`.

## Fresh-evaluation reliability (frozen manifest rev4; task 1008)

Twelve never-executed questions, frozen candidate, C=0 and strength-matched in-span random
controls. Primary semantic success = answer + persistent identity + coherence; formatting
separate. Descriptive n/N on a paired convenience sample -- NOT a population estimate.

In [7]:
def eval_dir(cond):
    return BATCH / f"out/2026-09-10_eval-{cond}/result.json"

# Durable per-row semantic adjudications (human-read full continuations, exact quotes in the
# record); the notebook aggregates the record and does not re-run heuristics.
ADJ = json.loads((BATCH / "slop/eval_fresh_adjudications.json").read_text())
print("adjudication provenance:", ADJ["provenance"][:80], "...")

CAP = 128  # the manifest's output cap; censoring = generation reached it
def table_line(r):
    n = len(json.loads((BATCH / f"out/2026-09-10_eval-{r['arm']}-{r['id']}/result.json")
                .read_text())["rows"][0]["generation"]["token_ids"])
    return f"| {r['id']}{' (no-op control)' if 'no-op' in r['notes'] else ''} | {r['arm']} "            f"| {r['answer']} | {r['identity']} | {r['coherence']} | {r['format']} | {n >= CAP} |"

rows = ADJ["rows"]
table = ["| question | arm | answer | identity | coherence | format | censored |",
         "|---|---|---|---|---|---|---|"]
counts, by_type, by_animal = {}, {}, {}
for r in rows:
    arm = r["arm"]
    counts.setdefault(arm, [0, 0])
    counts[arm][1] += 1
    primary = (r["answer"] == "pass" and r["identity"] == "pass" and r["coherence"] == "pass")
    counts[arm][0] += int(primary)
    # censoring derived from the saved token count vs the configured 128 cap (not from notes)
    n_tok = len(json.loads((BATCH / f"out/2026-09-10_eval-{r['arm']}-{r['id']}/result.json")
                .read_text())["rows"][0]["generation"]["token_ids"])
    censored = n_tok >= 128
    table.append(table_line(r))
    if arm == "candidate-C1.5":
        qt = r["id"].split("-")[0]; an = r["id"].rsplit("-", 1)[-1]
        bt = by_type.setdefault(qt, [0, 0]); bt[1] += 1; bt[0] += int(primary)
        ba = by_animal.setdefault(an, [0, 0]); ba[1] += 1; ba[0] += int(primary)
for arm, (k, n) in counts.items():
    table.append(f"| **{arm} PRIMARY** | | | | | | **{k}/{n}** |")
for tn, (k, n) in by_type.items():
    table.append(f"| **candidate by type: {tn}** | | | | | | **{k}/{n}** |")
for an, (k, n) in by_animal.items():
    table.append(f"| **candidate by animal: {an}** | | | | | | **{k}/{n}** |")
display(Markdown("\n".join(table)))

# Actual per-condition perturbation norms: candidate vs matched random (same C is not
# magnitude matching; the comparison is shown so any mismatch is visible).
import glob as _glob
norm_table = ["| question | candidate PREFILL norm | random PREFILL norm | candidate full-decode total | random full-decode total |", "|---|---|---|---|---|"]
for qp in sorted(_glob.glob(str(BATCH / "out/2026-09-10_eval-candidate-C1.5-*"))):
    qid = qp.split("eval-candidate-C1.5-")[1]
    cn = json.loads((BATCH / f"out/2026-09-10_eval-candidate-C1.5-{qid}/result.json").read_text())
    rn = json.loads((BATCH / f"out/2026-09-10_eval-random-C1.5-{qid}/result.json").read_text())
    crec = list(cn["rows"][0]["intervention_record"].values())[0]
    rrec = list(rn["rows"][0]["intervention_record"].values())[0]
    norm_table.append(f"| {qid} | {crec['perturbation_norm']:.3f} | {rrec['perturbation_norm']:.3f} "
                      f"| {crec['total_applied_norm']:.1f} | {rrec['total_applied_norm']:.1f} |")
display(Markdown("Perturbation norms: the PREFILL norm is the prefill-window edit magnitude; "
                 "total applied norm (prefill+decode) sums every applied edit across all cached "
                 "decode steps (candidate vs matched random at the same C; same C is not a "
                 "magnitude-match claim):\n\n" + "\n".join(norm_table)))

adjudication provenance: Per-row semantic adjudications by PI[claude] from direct reading of the full con ...


| question | arm | answer | identity | coherence | format | censored |
|---|---|---|---|---|---|---|
| name-N1-dog | candidate-C1.5 | pass | pass | pass | pass | False |
| name-N1-ant | candidate-C1.5 | ambiguous | fail | fail | fail | True |
| name-N2-dog | candidate-C1.5 | pass | pass | pass | pass | False |
| name-N2-ant | candidate-C1.5 | ambiguous | fail | fail | fail | True |
| legs-L1-dog | candidate-C1.5 | pass | pass | pass | pass | False |
| legs-L1-ant | candidate-C1.5 | pass | pass | pass | pass | False |
| legs-L2-dog | candidate-C1.5 | pass | pass | pass | pass | False |
| legs-L2-ant | candidate-C1.5 | pass | pass | pass | pass | False |
| prop-P1-spinneret-dog | candidate-C1.5 | fail | pass | fail | pass | False |
| prop-P1-spinneret-ant | candidate-C1.5 | fail | pass | fail | pass | True |
| prop-P2-liveyoung-dog | candidate-C1.5 | fail | pass | fail | pass | False |
| prop-P2-liveyoung-ant (no-op control) | candidate-C1.5 | pass | fail | fail | pass | False |
| name-N1-dog | random-C1.5 | fail | fail | fail | pass | False |
| name-N1-ant | random-C1.5 | fail | fail | fail | fail | True |
| name-N2-dog | random-C1.5 | fail | fail | pass | pass | False |
| name-N2-ant | random-C1.5 | fail | fail | fail | fail | True |
| legs-L1-dog | random-C1.5 | pass | fail | pass | pass | False |
| legs-L1-ant | random-C1.5 | fail | fail | pass | pass | False |
| legs-L2-dog | random-C1.5 | pass | fail | pass | pass | False |
| legs-L2-ant | random-C1.5 | fail | fail | pass | pass | False |
| prop-P1-spinneret-dog | random-C1.5 | fail | fail | fail | fail | True |
| prop-P1-spinneret-ant | random-C1.5 | fail | fail | fail | fail | True |
| prop-P2-liveyoung-dog | random-C1.5 | pass | pass | pass | fail | False |
| prop-P2-liveyoung-ant (no-op control) | random-C1.5 | pass | fail | pass | pass | False |
| **candidate-C1.5 PRIMARY** | | | | | | **6/12** |
| **random-C1.5 PRIMARY** | | | | | | **1/12** |
| **candidate by type: name** | | | | | | **2/4** |
| **candidate by type: legs** | | | | | | **4/4** |
| **candidate by type: prop** | | | | | | **0/4** |
| **candidate by animal: dog** | | | | | | **4/6** |
| **candidate by animal: ant** | | | | | | **2/6** |

Perturbation norms: the PREFILL norm is the prefill-window edit magnitude; total applied norm (prefill+decode) sums every applied edit across all cached decode steps (candidate vs matched random at the same C; same C is not a magnitude-match claim):

| question | candidate PREFILL norm | random PREFILL norm | candidate full-decode total | random full-decode total |
|---|---|---|---|---|
| legs-L1-ant | 18.928 | 21.808 | 568.1 | 964.5 |
| legs-L1-dog | 21.801 | 29.251 | 555.6 | 1008.1 |
| legs-L2-ant | 18.583 | 21.781 | 456.0 | 1186.8 |
| legs-L2-dog | 21.940 | 29.672 | 576.2 | 1062.3 |
| name-N1-ant | 19.078 | 21.758 | 1324.6 | 1754.0 |
| name-N1-dog | 19.764 | 27.989 | 461.7 | 935.5 |
| name-N2-ant | 18.705 | 21.079 | 1321.8 | 1547.9 |
| name-N2-dog | 21.239 | 28.884 | 459.4 | 1177.1 |
| prop-P1-spinneret-ant | 18.385 | 21.576 | 1203.5 | 1692.4 |
| prop-P1-spinneret-dog | 21.315 | 30.111 | 804.0 | 1966.1 |
| prop-P2-liveyoung-ant | 18.470 | 21.953 | 701.0 | 1253.6 |
| prop-P2-liveyoung-dog | 18.942 | 28.082 | 910.2 | 1380.3 |

### Example adjudication quotes

In [8]:
ex = next(r for r in ADJ["rows"] if r["id"] == "prop-P1-spinneret-ant")
display(Markdown("Failed cell `prop-P1-spinneret-ant` (fabricated anatomy):\n\n> " +
                 "\n\n> ".join(ex["quotes"])))
ex2 = next(r for r in ADJ["rows"] if r["id"] == "name-N1-ant")
display(Markdown("Failed cell `name-N1-ant` (repetition loop, ambiguous answer):\n\n> " +
                 "\n\n> ".join(ex2["quotes"])))

Failed cell `prop-P1-spinneret-ant` (fabricated anatomy):

> 1. Yes, ants have spinnerets.

Failed cell `name-N1-ant` (repetition loop, ambiguous answer):

> 1. Ant
2. Bee
3. Honeybee

## Paired fresh-set demonstrations (candidate cells, verbatim)

In [9]:
for cond, title in (("candidate-C1.5-legs-L1-dog", "Fresh legs, dog (digit + identity + coherent)"),
                    ("candidate-C1.5-prop-P1-spinneret-dog", "Fresh property, dog (source-bound answer: FAILS)")):
    row = json.loads(eval_dir(cond).read_text())["rows"][0]
    data = json.loads(eval_dir(cond).read_text())
    n = len(row["generation"]["token_ids"])
    display(Markdown(f"### {title} :: {cond}"))
    display(Markdown(f"Source (`repr`):\n\n```python\n{data['source_prompt']!r}\n```\n\n"
                     f"First 32 of {n} tokens:\n\n```text\n{TOK.decode(row['generation']['token_ids'][:32])}\n```\n\n"
                     f"Full continuation ({n} tokens):\n\n```text\n{row['generation']['text']}\n```"))

### Fresh legs, dog (digit + identity + coherent) :: candidate-C1.5-legs-L1-dog

Source (`repr`):

```python
'Question: How many legs does the animal known for spinning webs to catch insects have?\nAnswer: '
```

First 32 of 66 tokens:

```text
4

The animal is a dog, a popular domesticated breed known for its loyalty and friendly demeanor. Dogs typically have a thick coat of fur that comes in
```

Full continuation (66 tokens):

```text
4

The animal is a dog, a popular domesticated breed known for its loyalty and friendly demeanor. Dogs typically have a thick coat of fur that comes in various colors and textures to suit different climates. They are excellent companions for families and often serve as working animals in roles such as hunting, herding, and assistance.<|im_end|>
```

### Fresh property, dog (source-bound answer: FAILS) :: candidate-C1.5-prop-P1-spinneret-dog

Source (`repr`):

```python
'Question: Does the animal known for spinning webs to catch insects have spinnerets?\nAnswer: '
```

First 32 of 88 tokens:

```text
1. Yes, the animal known for spinning webs to catch insects has spinnerets.

The dog is a domesticated canine breed that is widely loved by humans
```

Full continuation (88 tokens):

```text
1. Yes, the animal known for spinning webs to catch insects has spinnerets.

The dog is a domesticated canine breed that is widely loved by humans for its loyalty and companionship. It typically has a thick coat of fur that comes in various colors and is often trained to perform tasks like herding or guarding. Dogs are social animals that communicate with their owners through barking, wagging tails, and expressive body language.<|im_end|>
```

## Mechanism findings and limits (linked sources; narrow facts)

- Measured pair agreement (L25, post-prefix): the SOURCE's maximum pair cosine is about
  +0.102; the donor prompts contain high-agreement subsets (dog about +0.854, ant about
  +0.584). The shared-prefix comparison (cos ~1.0) is a pipeline consistency check, not
  semantic validation. Details: batchwork `slop/2026-09-10_cross_token_cosine.md`.
- Pair selector: tested at rank 1, L25, C=1 and C=2 against a per-condition matched-random
  control; did not yield coherent replacement. ONE random draw per condition cannot establish
  equivalence or general nonseparation. Brief: batchwork
  `slop/2026-09-10_causal_selector_brief.md`.
- Donor-answer leakage: remains UNTESTED (not "attributable").
- Synchronized donor: a DIFFERENT shared_replace equation at L20 C1.5 (not the recovered
  span-correction candidate). Reduced distortion observed (correct eight-legged description
  where frozen invents four legs and a tail; natural end where frozen loops); NO target
  transfer. Brief: batchwork `slop/2026-09-10_synchronized_donor_brief.md`.
- Random control at C=2 in the replay also changed some dog leg answers (with wrong
  identity); the candidate-vs-random comparisons here are at matched C=1.5. No broader
  generalization from either.
- Limits: property fails at C=1.5; four adaptation items remain untested (recovery decision
  doc, batchwork `slop/2026-09-10_recovery_decision.md`).

In [10]:
display(Markdown("Sources: [cross-token report](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_cross_token_cosine.md) | "
                 "[recovery decision](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_recovery_decision.md) | "
                 "[frozen manifest](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_eval_manifest.md) | "
                 "[adjudication record](/workspace/2026/suppressed-activations-batchwork/slop/eval_fresh_adjudications.json) | "
                 "[selector brief](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_causal_selector_brief.md) | "
                 "[synchronized brief](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_synchronized_donor_brief.md)"))

Sources: [cross-token report](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_cross_token_cosine.md) | [recovery decision](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_recovery_decision.md) | [frozen manifest](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_eval_manifest.md) | [adjudication record](/workspace/2026/suppressed-activations-batchwork/slop/eval_fresh_adjudications.json) | [selector brief](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_causal_selector_brief.md) | [synchronized brief](/workspace/2026/suppressed-activations-batchwork/slop/2026-09-10_synchronized_donor_brief.md)

## Not demonstrated here# %% [markdown]
## Not demonstrated here

Cross-question persistence, readout calibration, and wider transfer are not established.
Readouts are verbatim and unvalidated. The 128-token cap is censoring: a capped continuation
is not evidence of natural completion. The 6/12 fresh-set rate is a descriptive fraction of a
paired convenience sample, not a population estimate or a solved mechanism.

## Artifact self-check (quote verification in-notebook; newline/link scan external)

In [11]:
# Every adjudication quote must be an exact substring of the saved generation it cites.
# (Markdown-newline and link-existence checks scan the executed notebook's outputs externally,
# after jupytext writes it - see justfile/notebook_verify.)
bad_quotes = [(r["id"], q[:40]) for r in ADJ["rows"]
              for q in r["quotes"]
              if q not in json.loads(
                  (BATCH / f"out/2026-09-10_eval-{r['arm']}-{r['id']}/result.json").read_text()
              )["rows"][0]["generation"]["text"]]
assert not bad_quotes, f"non-substring quotes: {bad_quotes}"
n_quotes = sum(len(r["quotes"]) for r in ADJ["rows"])
print(f"QUOTE CHECK PASS: all {n_quotes} quotes are exact substrings of their saved generations")

QUOTE CHECK PASS: all 32 quotes are exact substrings of their saved generations


## Full-deliverable self-check

In [12]:
# 1) censoring column matches saved token counts; 2) every markdown link in this notebook's
# own source points to an existing file; 3) aggregates derive from rows.
bad_cens = []
for r in ADJ["rows"]:
    n = len(json.loads((BATCH / f"out/2026-09-10_eval-{r['arm']}-{r['id']}/result.json")
                .read_text())["rows"][0]["generation"]["token_ids"])
    # the displayed censoring flag is derived from token counts; verify against the artifact
    if (n >= CAP) != any(f"| {r['id']}" in ln and "| True |" in ln for ln in [table_line(r)]):
        bad_cens.append((r["id"], r["arm"], n))
import re as _re2
slop_src = Path("/workspace/2026/suppressed-activations/slop/2026-09-10_span_correction_evidence.py")
src_links = _re2.findall(r"\]\((/[^)]+)\)", slop_src.read_text())
import os
broken = [l for l in src_links if not os.path.exists(l)]
assert not bad_cens, f"censoring mismatch: {bad_cens}"
assert not broken, f"broken source links: {broken}"
print(f"FULL SELF-CHECK PASS: censoring consistent with saved token counts; "
      f"{len(src_links)} absolute links exist; aggregates derived from rows")

FULL SELF-CHECK PASS: censoring consistent with saved token counts; 6 absolute links exist; aggregates derived from rows
